<a href="https://colab.research.google.com/github/EstefaniaS5/Proyecto-Buscaminas---Grupo-1/blob/main/Python%20-%20Proyecto_Buscaminas_Grupo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files #archivos google descargar

# CONFIGURACIÓN
FILAS = 16
COLS = 16
N_MINAS = 40

# 1. tablero vacío
def crear_tablero():
    return [[0] * COLS for _ in range(FILAS)]

# 2. Coloca minas aleatoriamente
def poner_minas(tablero):
    posiciones = random.sample(range(FILAS * COLS), N_MINAS)
    for pos in posiciones:
        fila = pos // COLS
        col = pos % COLS
        tablero[fila][col] = -1
    return tablero

# 3. Calcula cuántas minas vecinas tiene cada celda
def calcular_vecinos(tablero):
    for f in range(FILAS):
        for c in range(COLS):
            if tablero[f][c] == -1:
                continue

            count = 0
            for df in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if df == 0 and dc == 0:
                        continue

                    nf, nc = f + df, c + dc
                    if 0 <= nf < FILAS and 0 <= nc < COLS:
                        if tablero[nf][nc] == -1:
                            count += 1

            tablero[f][c] = count
    return tablero

# 4. Muestra tablero en consola
def imprimir_tablero(tablero):
    simbolos = {-1: " * "}
    print("   " + " ".join(f"{c:2}" for c in range(COLS)))
    print("   " + "───" * COLS)

    for f, fila in enumerate(tablero):
        fila_str = []
        for val in fila:
            fila_str.append(simbolos.get(val, f"{val:2} "))
        print(f"{f:2}|" + "".join(fila_str))

# 5. Genera bloque de memoria para MARIE
def generar_bloque_marie(tablero):
    lineas = ["; ── Tablero Buscaminas 16x16 para MARIE ──"]

    for f in range(FILAS):
        for c in range(COLS):
            valor = tablero[f][c]

            # En MARIE la mina se guardará como 4095
            if valor == -1:
                valor_marie = 4095
            else:
                valor_marie = valor

            etiqueta = f"C{f:02d}{c:02d}"
            lineas.append(f"{etiqueta}, DEC {valor_marie}")

    return "\n".join(lineas)

# 6. Guarda imagen PNG del tablero
def guardar_png_tablero(tablero, nombre_archivo="tablero_buscaminas.png"):
    colores = {
        -1: "#000000",   # mina
         0: "#d9d9d9",   # 0
         1: "#4da6ff",   # 1
         2: "#5cd65c",   # 2
         3: "#ff4d4d",   # 3
         4: "#1f4e79",   # 4
         5: "#990000",   # 5
         6: "#33cccc",   # 6
         7: "#9933cc",   # 7
         8: "#ffffff"    # 8
    }

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(0, COLS)
    ax.set_ylim(0, FILAS)
    ax.set_aspect('equal')
    ax.axis('off')

    for f in range(FILAS):
        for c in range(COLS):
            val = tablero[f][c]

            rect = patches.Rectangle(
                (c, FILAS - 1 - f), 1, 1,
                facecolor=colores[val],
                edgecolor="black",
                linewidth=1
            )
            ax.add_patch(rect)

            # Muestra mina o número en la imagen
            if val == -1:
                ax.text(
                    c + 0.5, FILAS - 1 - f + 0.5, "*",
                    ha='center', va='center',
                    fontsize=12, color='white', fontweight='bold'
                )
            elif val != 0:
                color_texto = "white" if val in [4, 5, 7] else "black"
                ax.text(
                    c + 0.5, FILAS - 1 - f + 0.5, str(val),
                    ha='center', va='center',
                    fontsize=11, color=color_texto, fontweight='bold'
                )

    plt.tight_layout()
    plt.savefig(nombre_archivo, dpi=200, bbox_inches='tight')
    plt.close()

# MAIN
tablero = crear_tablero()
tablero = poner_minas(tablero)
tablero = calcular_vecinos(tablero)

print("TABLERO BUSCAMINAS 16x16")
imprimir_tablero(tablero)

print("\nBLOQUE MARIE COMPLETO (256 celdas)")
bloque = generar_bloque_marie(tablero)
print(bloque)

# Guarda archivo ASM
with open("tablero_marie.asm", "w") as f:
    f.write(bloque)
print("\nArchivo 'tablero_marie.asm' generado correctamente.")

# Guarda imagen PNG
guardar_png_tablero(tablero)
print("Archivo 'tablero_buscaminas.png' generado correctamente.")

# descomentar estas líneas para descargar
from google.colab import files
#files.download("tablero_marie.asm")
#files.download("tablero_buscaminas.png")

TABLERO BUSCAMINAS 16x16
    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15
   ────────────────────────────────────────────────
 0| 0  2  *  2  1  1  2  1  1  0  0  1  2  2  1  0 
 1| 0  2  *  2  1  *  2  *  2  1  0  1  *  *  1  0 
 2| 0  1  1  1  1  1  2  2  *  2  1  2  3  3  1  0 
 3| 0  1  1  1  0  0  0  2  3  *  2  2  *  2  1  1 
 4| 1  3  *  2  0  0  0  1  *  3  *  3  2  2  *  1 
 5| 2  *  *  2  0  0  0  1  1  2  2  *  1  1  1  1 
 6| *  4  2  1  0  0  0  0  0  0  1  1  1  0  0  0 
 7| *  3  0  0  0  0  0  1  2  2  1  0  0  0  0  0 
 8| *  2  0  0  0  1  1  2  *  *  2  1  0  0  0  0 
 9| 2  2  0  0  0  1  *  2  2  3  *  1  0  0  0  0 
10| *  1  0  0  0  2  2  2  0  1  1  2  1  1  0  0 
11| 1  1  1  1  1  1  *  1  0  0  0  1  *  1  0  0 
12| 0  0  2  *  2  1  2  2  1  0  0  1  1  2  1  1 
13| 1  1  4  *  3  1  2  *  1  0  0  0  0  2  *  2 
14| 2  *  4  *  3  3  *  3  2  1  1  1  1  3  *  3 
15| 2  *  3  2  *  3  *  2  1  *  1  1  *  2  2  * 

BLOQUE MARIE COMPLETO (256 celdas)
; ──